# **EDA Notebook**



---
## 0. Setup Environment

In [ ]:
# Do not modify this code
%pip install uts_36106_core==0.0.3

from uts_36106_core.folders import *
from uts_36106_core.ipyrenders import *
import os

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.mount_gdrive()
at.create_folders()
if at.is_colab:
  os.chdir(f'{at.folder_path}')

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
# <Student to fill this section>
group_name = ""
student_name = ""
student_id = ""

In [ ]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [ ]:
# No additional packages: pandas, numpy, scipy and altair are available on Colab.

### 0.b Import Packages

In [ ]:
import pandas as pd
import altair as alt
import numpy as np
from scipy.stats import kruskal, norm, spearmanr
from IPython.display import display

In [ ]:
# Utils

def column_profile(df):
    """One row per column: type, missing count and share, distinct values."""
    return pd.DataFrame({
        "type": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_share": df.isna().mean().round(4),
        "distinct": df.nunique(),
    })


def past_and_future_change(balances, account_col, date_col, balance_col, horizon):
    """Change over the `horizon` month-ends before and after each month-end, per account."""
    ordered = balances.sort_values([account_col, date_col])
    grouped = ordered.groupby(account_col)[balance_col]
    out = ordered.copy()
    out["past_change"] = ordered[balance_col] - grouped.shift(horizon)
    out["future_change"] = grouped.shift(-horizon) - ordered[balance_col]
    return out


def spearman_summary(x, y, label):
    """Spearman correlation with a Fisher-z 95% interval and a size label."""
    mask = x.notna() & y.notna()
    rho, p_value = spearmanr(x[mask], y[mask])
    n = int(mask.sum())
    half_width = 1.96 / np.sqrt(n - 3)
    low, high = np.tanh(np.arctanh(rho) - half_width), np.tanh(np.arctanh(rho) + half_width)
    size = abs(rho)
    magnitude = "negligible" if size < 0.1 else "small" if size < 0.3 else "medium" if size < 0.5 else "large"
    return pd.Series({"feature": label, "n": n, "rho": round(rho, 4), "ci_low": round(low, 4),
                      "ci_high": round(high, 4), "p_value": p_value, "magnitude": magnitude})


def holm_adjust(p_values):
    """Holm step-down adjustment of a list of p-values (family-wise error control)."""
    order = np.argsort(p_values)
    m = len(p_values)
    adjusted = np.empty(m)
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (m - rank) * p_values[index])
        adjusted[index] = min(running, 1.0)
    return adjusted


def kruskal_epsilon(df, value_col, group_col):
    """Kruskal-Wallis H test across groups with epsilon-squared effect size."""
    groups = [g[value_col].dropna().to_numpy() for _, g in df.groupby(group_col)]
    h_stat, p_value = kruskal(*groups)
    n, k = sum(len(g) for g in groups), len(groups)
    epsilon_sq = (h_stat - k + 1) / (n - k)
    return pd.Series({"H": round(h_stat, 2), "groups": k, "n": n, "p_value": p_value,
                      "epsilon_squared": round(epsilon_sq, 6)})


def wilson_interval(successes, n):
    """95% Wilson interval for a proportion."""
    z = 1.96
    p = successes / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half


def qq_frame(values):
    """Sample quantiles against normal quantiles (all points up to 500 values, else 199 levels).

    Robust standardisation (median, IQR / 1.349) keeps a few extreme values from flattening the plot;
    for normal data it equals the usual standardisation."""
    values = pd.Series(values).dropna()
    n = len(values)
    levels = (np.arange(1, n + 1) - 0.5) / n if n <= 500 else np.linspace(0.005, 0.995, 199)
    scale = (values.quantile(0.75) - values.quantile(0.25)) / 1.349
    scale = scale if scale > 0 else values.std()  # IQR is 0 when most values are equal
    z = (values - values.median()) / scale
    return pd.DataFrame({"theoretical": norm.ppf(levels), "sample": np.quantile(z, levels)})


def distribution_charts(values, label, n_bins, tail):
    """Histogram (the `tail` share at each end not drawn) beside a normal QQ plot, with shape statistics."""
    values = pd.Series(values).dropna()
    low, high = values.quantile(tail), values.quantile(1 - tail)
    shown = values[(values >= low) & (values <= high)]
    counts, edges = np.histogram(shown, bins=n_bins)
    hist = alt.Chart(pd.DataFrame({"start": edges[:-1], "end": edges[1:], "count": counts})).mark_bar(
        color="#2a78d6").encode(x=alt.X("start:Q", bin="binned", title=label), x2="end:Q",
                                y=alt.Y("count:Q", title="Count")).properties(width=300, height=220)
    diagonal = pd.DataFrame({"theoretical": [-3, 3], "sample": [-3, 3]})
    qq = (alt.Chart(qq_frame(values)).mark_point(color="#2a78d6", size=14).encode(
            x=alt.X("theoretical:Q", title="Normal quantile"), y=alt.Y("sample:Q", title="Robust-standardised sample quantile"))
          + alt.Chart(diagonal).mark_line(color="#898781", strokeDash=[4, 4]).encode(x="theoretical:Q", y="sample:Q")
          ).properties(width=300, height=220)
    print(f"{label}: n = {len(values)}, skewness = {values.skew():.2f}, excess kurtosis = {values.kurt():.2f}, "
          f"histogram omits {len(values) - len(shown)} values outside the {tail:.1%} to {1 - tail:.1%} quantiles")
    return alt.hconcat(hist, qq)

In [ ]:
# Constants
# <HORIZON> is a provisional working horizon (months ahead) used where one horizon is needed;
# B.2 examines the horizons in LAGS before any horizon is chosen.
HORIZON = 3
LAGS = [1, 3, 6]
DROP_LEVELS = [0.05, 0.10, 0.20]   # relative drops examined in B.4; none is chosen here

---
## B. Data Understanding

**Approach.** This notebook studies the monthly balance history: first its quality (B.1), then three questions that the forecast design depends on: how far ahead a forecast can still be informative (B.2), whether account types behave differently (B.3), and how often large balance drops happen and whether anything in this table signals them (B.4). Each finding is stated only from the output directly above it; the forecast horizon, the accounts in scope and the business use are decided only after all three EDA notebooks.

In [ ]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "account_balance_history.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [ ]:
df["balance_date"] = pd.to_datetime(df["balance_date"])
print("Rows, columns:", df.shape)
print("Accounts:", df["account_id"].nunique(), "| month-ends:", df["balance_date"].nunique(),
      "| from", df["balance_date"].min().date(), "to", df["balance_date"].max().date())
print("Month-ends per account:", df.groupby("account_id").size().value_counts().to_dict())
print("Duplicated (account_id, balance_date):", df.duplicated(["account_id", "balance_date"]).sum())
display(column_profile(df))

ordered = df.sort_values(["account_id", "balance_date"])
diff = ordered.groupby("account_id")["balance"].diff()
consistent = np.isclose(diff, ordered["balance_change"], atol=0.01)[diff.notna()]
print("balance_change equals the month-on-month difference (within $0.01):", round(consistent.mean(), 4),
      "of", int(diff.notna().sum()), "checkable rows")

display(df["balance"].describe().round(2).to_frame("balance ($)"))
print("Zero balances:", (df["balance"] == 0).sum(), "| negative balances:", (df["balance"] < 0).sum())
print("Balance range by balance_band ($):")
display(df.groupby("balance_band")["balance"].agg(["min", "max", "size"]).sort_values("min").round(0))
print("stress_indicator values (share of rows):")
display(df["stress_indicator"].value_counts(normalize=True).sort_index().round(4).to_frame("share").T)

# Complementary load: account attributes to check history against the account's life
accounts = pd.read_csv(at.folder_path / "accounts.csv", usecols=["account_id", "account_type", "open_date",
                                                                  "account_status", "current_balance"])
accounts["open_date"] = pd.to_datetime(accounts["open_date"])
with_accounts = df.merge(accounts, on="account_id", how="left")
before_open = with_accounts["balance_date"] < with_accounts["open_date"]
print("Month-ends dated before the account's open_date:", int(before_open.sum()), f"({before_open.mean():.1%})")
last = with_accounts[with_accounts["balance_date"] == with_accounts["balance_date"].max()]
print("Last month-end balance equals accounts.current_balance (within $1):",
      round(np.isclose(last["balance"], last["current_balance"], atol=1).mean(), 4))
print("Spread of the monthly change by balance quintile ($, standard deviation):")
display(ordered.groupby(pd.qcut(ordered["balance"], 5))["balance_change"].std().round(0).to_frame("std of monthly change"))

In [ ]:
monthly = (df.groupby("balance_date")["balance"].agg(mean="mean", median="median").reset_index()
           .melt("balance_date", var_name="statistic", value_name="balance"))
alt.Chart(monthly).mark_line(point=True).encode(
    x=alt.X("balance_date:T", title="Month-end"),
    y=alt.Y("balance:Q", title="Balance across accounts ($)", scale=alt.Scale(zero=False)),
    color=alt.Color("statistic:N", title=None, scale=alt.Scale(range=["#2a78d6", "#898781"])),
    tooltip=["balance_date:T", "statistic", alt.Tooltip("balance:Q", format=".0f")],
).properties(width=620, height=240)

In [ ]:
dataset_insights = """
Size and coverage. 1850976 rows and 8 columns: 38562 accounts, each with all 48 month-ends
from 2022-01-31 to 2025-12-31. No missing values and no duplicated account-month.

Consistency. <balance_change> equals the month-on-month difference of balance (within $0.01) in
99.86% of the 1812414 rows where it can be checked; the remaining 0.14% disagree, so any later
step recomputes the change from the balances rather than trusting the column.

Balances. From $0 to $3.65 million; median $208402, mean $405201 (strong right skew).
2584 zero balances, none negative. <balance_band> is a fixed dollar range: Low up to $1000,
Medium to $10000, High to $50000, Premium to $250000, Private Banking above; 85% of rows are
Premium or Private Banking. The chart shows the mean balance rising slowly from about $400000
to $410000 over the four years, and the median from about $204000 to $212000.

<stress_indicator> takes 16 values from 0 to 95 (50.6% at 0, 27.5% at 10); it is a score, not a
flag, and the schema does not define it.

Issues found.
1. 278116 month-ends (15.0%) are dated before the account's <open_date> yet carry a balance:
   that is not history of the account and must be handled before any modelling.
2. <accounts.current_balance> matches the last month-end balance for only 0.01% of accounts, so
   that field is not consistent with this history.
3. The spread of the monthly change grows with the balance (standard deviation $2327 in the
   lowest balance quintile, $7715 in the highest). An average dollar error will be driven by
   large accounts, so results must also be read by balance segment.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### B.2 Explore Feature of Interest `future_change` at 1, 3 and 6 months

In [ ]:
# Change from each month-end to the month-end 1, 3 and 6 months later, per account
changes_by_lag = {lag: past_and_future_change(df, "account_id", "balance_date", "balance", lag) for lag in LAGS}
alt.vconcat(*[distribution_charts(changes_by_lag[lag]["future_change"], f"Balance change over {lag} month(s) ($)", 40, 0.005)
              for lag in LAGS])

In [ ]:
# How much does "the balance stays the same" miss at each horizon, and does the recent movement
# say anything about the next one?
rows, tests = [], []
for lag in LAGS:
    frame = changes_by_lag[lag]
    future = frame["future_change"].dropna()
    rows.append({"horizon_months": lag, "examples": len(future),
                 "persistence_mae": round(future.abs().mean(), 1),
                 "mean_change": round(future.mean(), 1),
                 "mae_if_mean_change_added": round((future - future.mean()).abs().mean(), 1)})
    tests.append(spearman_summary(frame["past_change"], frame["future_change"], f"past vs next {lag} months"))
horizon_table = pd.DataFrame(rows)
test_table = pd.DataFrame(tests).set_index("feature")
test_table["p_holm"] = holm_adjust(test_table["p_value"].to_numpy())
display(horizon_table)
display(test_table)

monthly_change = ordered.assign(prev=ordered.groupby("account_id")["balance_change"].shift(1))
print("Correlation of one month's change with the previous month's change:",
      round(monthly_change["balance_change"].corr(monthly_change["prev"]), 4))

In [ ]:
mae_chart = alt.Chart(horizon_table).mark_bar(color="#2a78d6").encode(
    x=alt.X("horizon_months:O", title="Months ahead"),
    y=alt.Y("persistence_mae:Q", title="Average miss of 'balance stays the same' ($)"),
    tooltip=["horizon_months", "persistence_mae", "mae_if_mean_change_added"],
).properties(width=260, height=240)
rho_table = test_table.reset_index().assign(horizon_months=LAGS)
rho_chart = (alt.Chart(rho_table).mark_rule(color="#898781").encode(
        x=alt.X("horizon_months:O", title="Months ahead"),
        y=alt.Y("ci_low:Q", title="Spearman rho, past vs next change", scale=alt.Scale(domain=[-0.1, 0.1])),
        y2="ci_high:Q")
    + alt.Chart(rho_table).mark_point(filled=True, color="#2a78d6").encode(
        x="horizon_months:O", y="rho:Q", tooltip=["feature", "rho", "ci_low", "ci_high"])
    + alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(color="#898781", strokeDash=[4, 4]).encode(y="y:Q")
    + alt.Chart(pd.DataFrame({"y": [0], "label": ["no relation"]})).mark_text(
        align="left", dx=4, dy=-7, color="#898781").encode(y="y:Q", x=alt.value(0), text="label:N")
).properties(width=260, height=240)
alt.hconcat(mae_chart, rho_chart)

In [ ]:
feature_1_insights = """
Question. How much does the bank's current assumption, "the balance stays the same", miss at
1, 3 and 6 months ahead, and does an account's recent movement say anything about its next one?

Miss of the current assumption. $3465 on average at 1 month, $6019 at 3 months and $8547 at
6 months (left chart). The miss grows close to the square root of the horizon (3 months is 1.74
times the 1-month miss, 6 months is 2.47 times; the square roots are 1.73 and 2.45), which is
what independent month-to-month changes produce. Adding the average change on top improves the
miss by at most 0.4%.

Test choice. The histograms are close to symmetric (skewness 0.10 to 0.24), but every QQ plot bends
away from the line in both tails (excess kurtosis 3.36 to 3.43): heavy tails, so the Spearman rank correlation between the change over
the past h months and over the next h months, with Holm correction across the three horizons.

Result. rho = 0.0008, 0.0030 and 0.0076 for 1, 3 and 6 months: all negligible (right chart). The
3- and 6-month values are statistically different from zero only because of the very large
number of examples (1.4 to 1.8 million). One month's change and the next have a correlation of
0.0006.

Reading. In this table, the recent movement of a balance carries practically no information about
its next movement at any of the three horizons. The horizon decision is not taken here; this is
one of the inputs for it.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `account_type`

In [ ]:
by_type = (past_and_future_change(df, "account_id", "balance_date", "balance", HORIZON)
           .merge(accounts[["account_id", "account_type"]], on="account_id").dropna(subset=["future_change"]))

# Examples per account type, and the shape of the change within each type
type_counts = by_type["account_type"].value_counts().rename_axis("account_type").reset_index(name="examples")
bars = alt.Chart(type_counts).mark_bar(color="#2a78d6").encode(
    x=alt.X("examples:Q", title="Examples"), y=alt.Y("account_type:N", title="Account type")).properties(width=300, height=220)
qq = pd.concat([qq_frame(group["future_change"]).assign(account_type=kind) for kind, group in by_type.groupby("account_type")])
diagonal = pd.DataFrame({"theoretical": [-3, 3], "sample": [-3, 3]})
qq_by_type = (alt.Chart(qq).mark_line().encode(
        x=alt.X("theoretical:Q", title="Normal quantile"),
        y=alt.Y("sample:Q", title=f"Robust-standardised {HORIZON}-month change quantile"),
        color=alt.Color("account_type:N", title="Account type"))
    + alt.Chart(diagonal).mark_line(color="#898781", strokeDash=[4, 4]).encode(x="theoretical:Q", y="sample:Q")
).properties(width=300, height=220)
print(by_type.groupby("account_type")["future_change"].agg(skewness="skew", excess_kurtosis=lambda s: s.kurt()).round(2))
alt.hconcat(bars, qq_by_type)

In [ ]:
# Do account types move differently over the working horizon?
summary = by_type.groupby("account_type").agg(
    accounts=("account_id", "nunique"), examples=("future_change", "size"), mean=("future_change", "mean"),
    std=("future_change", "std"), median=("future_change", "median"),
    zero_change_share=("future_change", lambda s: (s == 0).mean())).round(4)
display(summary)
type_test = kruskal_epsilon(by_type, "future_change", "account_type")
display(type_test.to_frame("Kruskal-Wallis, change over the next HORIZON months by account type"))

In [ ]:
quantiles = (by_type.groupby("account_type")["future_change"]
             .quantile([0.05, 0.25, 0.5, 0.75, 0.95]).unstack().reset_index())
quantiles.columns = ["account_type", "p05", "p25", "p50", "p75", "p95"]
base = alt.Chart(quantiles).encode(y=alt.Y("account_type:N", title="Account type"))
(base.mark_rule(color="#898781").encode(x=alt.X("p05:Q", title=f"Balance change over the next {HORIZON} months ($), 5th to 95th percentile"), x2="p95:Q")
 + base.mark_bar(color="#2a78d6", height=14).encode(x="p25:Q", x2="p75:Q")
 + base.mark_tick(color="white", thickness=2, size=14).encode(x="p50:Q")
).properties(width=560, height=220)

In [ ]:
feature_2_insights = """
Question. Do account types move differently over the working horizon (3 months)? The answer
decides whether some types, Term Deposits in particular, should be forecast separately or left out.

Description. Mean 3-month change between $654 (Business) and $717 (Youth); standard deviation
between $8537 and $8587 for every type; months with no change at all are 0.05% or less for
every type, Term Deposit included. The chart shows the same spread for all six types.

Test choice. Six groups whose QQ curves lie on top of each other and bend away from the line in
both tails (excess kurtosis 3.23 to 3.75), so Kruskal-Wallis, with epsilon squared as the
effect size.

Result. H = 11.97, p = 0.035, epsilon squared = 0.000004: a difference detectable only because of
1735290 examples, and of negligible size.

Reading. In this data a Term Deposit or Business balance changes as often and by as much as a
Savings or Transaction balance, which is unlike real term deposits. The account type label does
not separate balance behaviour here.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `relative_change` (large drops)

In [ ]:
drops = past_and_future_change(df, "account_id", "balance_date", "balance", HORIZON).dropna(subset=["future_change"])
drops = drops[drops["balance"] > 0].copy()
drops["relative_change"] = drops["future_change"] / drops["balance"]

distribution_charts(drops["relative_change"], f"Relative balance change over {HORIZON} months", 40, 0.005)

In [ ]:
# How often does a balance fall sharply over the working horizon, and does <stress_indicator> signal it?
level_rows = []
for level in DROP_LEVELS:
    hits = int((drops["relative_change"] <= -level).sum())
    low, high = wilson_interval(hits, len(drops))
    level_rows.append({"drop_of_at_least": f"{level:.0%}", "examples": hits, "share": round(hits / len(drops), 4),
                       "ci_low": round(low, 4), "ci_high": round(high, 4)})
display(pd.DataFrame(level_rows))

band_rows = []
for band, group in drops.groupby("balance_band"):
    hits = int((group["relative_change"] <= -0.10).sum())
    low, high = wilson_interval(hits, len(group))
    band_rows.append({"balance_band": band, "examples": len(group), "median_balance": round(group["balance"].median(), 0),
                      "share_drop_10pct": round(hits / len(group), 4), "ci_low": round(low, 4), "ci_high": round(high, 4),
                      "mean_stress": round(group["stress_indicator"].mean(), 2)})
band_table = pd.DataFrame(band_rows).sort_values("median_balance")
display(band_table)
stress_test = spearman_summary(drops["stress_indicator"], drops["relative_change"], "stress_indicator vs relative change")
display(stress_test.to_frame("Spearman, stress_indicator vs relative change over the next HORIZON months"))

In [ ]:
overall_share = float((drops["relative_change"] <= -0.10).mean())
ref = pd.DataFrame({"x": [overall_share], "label": [f"all month-ends: {overall_share:.2%}"]})
base = alt.Chart(band_table).encode(y=alt.Y("balance_band:N", title="Balance band", sort=list(band_table["balance_band"])))
(base.mark_rule(color="#898781").encode(x=alt.X("ci_low:Q", title=f"Share of month-ends followed by a drop of 10% or more within {HORIZON} months"), x2="ci_high:Q")
 + base.mark_point(filled=True, color="#e34948").encode(x="share_drop_10pct:Q",
       tooltip=["balance_band", "examples", "share_drop_10pct", "ci_low", "ci_high", "mean_stress"])
 + alt.Chart(ref).mark_rule(color="#898781", strokeDash=[4, 4]).encode(x="x:Q")
 + alt.Chart(ref).mark_text(align="left", dx=4, color="#898781").encode(x="x:Q", y=alt.value(0), text="label:N")
).properties(width=520, height=200)

In [ ]:
feature_n_insights = """
Question. How often does a balance fall sharply within 3 months, where, and does <stress_indicator>
signal it in advance?

Frequency. Of 1732960 month-ends with a positive balance, 10.2% are followed by a drop of 5% or
more within 3 months, 4.25% by 10% or more and 1.45% by 20% or more.

Where. Drops of 10% or more are concentrated in small balances (chart): 37.2% of month-ends in the
Low band and 39.7% in Medium, 17.5% in High, 3.0% in Premium and 0.05% in Private Banking. This
follows from B.1: the dollar spread of changes grows much more slowly than the balance, so the
same size of movement is a large share of a small balance.

Signal. <stress_indicator> is higher on average in the Low band (32.75, against 7.5 to 8.0
elsewhere), but its Spearman correlation with the relative change that follows is 0.001 (95%
interval -0.0004 to 0.0025): negligible. Test choice: a discrete score against a ratio whose QQ plot
bends steeply in both tails (skewness 1025.24, driven by a few near-zero balances; the histogram
omits the outer 1%), so a rank correlation.

Reading. Large relative drops are frequent only among accounts with small balances, and nothing
in this table flags them before they happen. Whether such drops are a useful business target is
decided after all three EDA notebooks.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

## C. Ethical Consideration of Features


In [ ]:
# Who sits in the lowest balance bands? A first look at whether low balances concentrate
# in particular customer groups (complementary load of customers).
customers = pd.read_csv(at.folder_path / "customers.csv", usecols=["customer_id", "age", "gender", "vulnerable_customer_flag"])
latest = df[df["balance_date"] == df["balance_date"].max()].merge(customers, on="customer_id", how="left")
display(latest.groupby("balance_band").agg(accounts=("account_id", "size"), mean_age=("age", "mean"),
                                           vulnerable_share=("vulnerable_customer_flag", "mean")).round(3))

In [ ]:
ethical_considerations = """
Balances and <stress_indicator> describe a customer's financial position, and the table above shows
that the lowest balance bands hold a larger share of vulnerable customers (5.8% in Low and 4.5% in
Medium, against 3.1% to 3.4% in Premium and Private Banking), at a similar mean age (about 51).

Any use that focuses on accounts with small balances, such as an early warning of large drops,
therefore reaches vulnerable customers more often. Action: such a list may only trigger support
(a contact, a hardship conversation), never a restriction, fee or credit decision, and results are
audited by customer group before use. The months recorded before an account opened (B.1) are not
real history; using them would describe new customers wrongly, so they must be handled before
modelling.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='ethical_considerations', value=ethical_considerations)